In [1]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

/Users/rahulprajapati/miniconda3/envs/langchain/lib/python3.10/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

In [3]:
model = ChatOpenAI()

In [4]:
# define the state

class BlogsState(TypedDict):
    title: str
    outline: str
    content: str

In [5]:
def create_outline(state: BlogsState):
    
    title = state['title']

    prompt = f"Generate the detailed outline for the blogs on the given topic {title}"

    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state


In [6]:
def create_blog(state: BlogsState):

    title = state['title']
    outline = state['outline']

    prompt= f'Generate the detailed blogs on the title - {title} by following outline \n {outline}'

    answer = model.invoke(prompt).content


    state['content'] = answer

    return state

In [7]:
# define the stateGraph

graph = StateGraph(BlogsState)

# add node
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# add edge
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

# comiple the graph
workflow = graph.compile()



In [10]:
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)
print(final_state)
print(final_state['outline'])
print('='*50)
print(final_state['content'])

{'title': 'Rise of AI in India', 'outline': "I. Introduction\n    A. Brief overview of artificial intelligence (AI)\n    B. Explanation of the rise of AI in India\n    C. Importance of AI in various industries in India\n\nII. Current State of AI in India\n    A. Overview of the current use of AI in India\n    B. Major players and companies involved in AI research and development in India\n    C. Key applications of AI in different sectors in India\n\nIII. Driving Factors for the Growth of AI in India\n    A. Government initiatives and policies supporting AI research and development\n    B. Presence of a skilled workforce in India for AI\n    C. Increasing demand for automation and efficiency in various industries\n    D. Growth of startups specializing in AI technology in India\n\nIV. Challenges and Opportunities for AI in India\n    A. Lack of infrastructure for AI research and development\n    B. Ethical and regulatory challenges in the use of AI in India\n    C. Opportunities for co